# Exercise 2 - Changes in High Elevation Regions

Many high-elevation regions are of critical importance for downstream water availability, natural hazards, and ecosystem health. However, many of them are difficult to access due to lack of roads, steep terrain, and extreme temperatures. We can use remote sensing data to monitor changes in such difficult environments.

### Goals
In this exercise, we will look at changes through time using distinct time points, differences between years, and trends. We will focus on Glacier Lakes as well as land cover and climate trends.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import datetime

In [ ]:
import ee
import geemap
project_name = ''
#The first time you use the earthengine module, you need to link your account credentials. Afterwards, your computer stores the authentication file
ee.Authenticate()
ee.Initialize()
#ee.Initialize(project=project_name)
print(ee.__version__)

We can quickly test to see if everything imported correctly:

In [ ]:
dem = ee.Image('USGS/SRTMGL1_003') #Load in the global SRTM elevation data set
xy = ee.Geometry.Point([86.9250, 27.9881]) #Define the location of interest
elev = dem.sample(xy, 30).first().get('elevation').getInfo() #Sample the data set at that point
print('Mount Everest elevation (m):', elev) #Print the result

## Changing Glacier Lakes

Throughout the Himalaya, glaciers are retreating and new lakes are forming. Monitoring how they are changing is critical to understanding flood and landslide risk. Let's take a look at one glacier lake and visualize it over multiple years. 

We can use the Landsat sensor to look at historical imagery:

In [ ]:
lake_center = ee.Geometry.Point([85.457806, 28.568519]) #One point near a glacier lake
landsat = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").filterBounds(lake_center) #Filter our data to that location

def apply_scale_factors(image): #Define a scale factor conversion as before
    optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
    return image.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)

In [ ]:
single_image = landsat.first() #Choose the first image in our collection
single_image = apply_scale_factors(single_image) #Apply the scale factors

visualization = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min':0, 'max':0.3} #Choose false-color bands

Map = geemap.Map(center=[28.568519, 85.457806], zoom=14) #Set up a map, this time more zoomed in
Map.addLayer(single_image, visualization) #Plot the map
Map.addLayer(lake_center) #Plot the center point

Map

To look further back in the past, we need to use a different Landsat sensor. Landsat 5 was running from the 1980s until ~2012, so is a good option for historical imagery. Let's load that in and visualize it. The description of the data can be found here: [https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LT05_C02_T1_L2](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LT05_C02_T1_L2) 

In [ ]:
landsat5 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2").filterBounds(lake_center) #Use the same point to filter the data

def apply_scale_factors_landsat5(image): #Use a different correction factor for the older Landsat data
    optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B6').multiply(0.00341802).add(149.0)
    return image.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)

In [ ]:
single_image_l5 = landsat5.filterDate('1998-06-01','2012-01-01').first() #Note that I filtered the timing to a summer image!
single_image_l5 = apply_scale_factors_landsat5(single_image_l5) #Apply the scale factors

visualization_l5 = {'bands': ['SR_B3', 'SR_B2', 'SR_B1'], 'min':0, 'max':0.5} #Choose false-color bands

Map = geemap.Map(center=[28.568519, 85.457806], zoom=14) #Set up a map, this time more zoomed in
Map.addLayer(single_image_l5, visualization_l5) #Plot the map
Map.addLayer(lake_center) #Plot the center point

Map

We can also visualize those two images side-by-side using geemap for easier comparison.

By moving the slider back and forth we can see how things have changed through time. Note that the lake on the left has grown significantly!

In [ ]:
left_layer = geemap.ee_tile_layer(single_image_l5, visualization_l5, "1998")
right_layer = geemap.ee_tile_layer(single_image, visualization, "2025")

Map = geemap.Map(center=[28.568519, 85.457806], zoom=14)
Map.split_map(left_layer, right_layer)
Map

This same approach can also be used to compare data over shorter times (e.g., summer vs winter). We will return to the split map later in this exercise to compare different data sets.

## Visualizing Long-Term Average Maps

We can use the **Earth Engine Reducer** tools ([https://developers.google.com/earth-engine/guides/reducers_intro](https://developers.google.com/earth-engine/guides/reducers_intro)) to do operations over _space_ or _time_. This is what we briefly did in the previous exercise to examine average rainfall through one year, but the reducer tools can be used for many other kinds of operations.

Let's take a look at vegetation data and examine long-term averages, statistics, and changes. We can start by loading in the MODIS NDVI data ([https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13A1](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13A1)), which gives us access to 25 years of data.

In [ ]:
def rescale(ic, scale, add=0):
    def r(image):
        return image.multiply(scale).add(add).set('system:time_start', image.get('system:time_start'))
    return ic.map(r)

modis = ee.ImageCollection("MODIS/061/MOD13A1").select('NDVI') #Choose only the NDVI data
modis = rescale(modis, 0.0001) #Rescale all of the images to the proper range
modis.first().getInfo()

Let's compute a few statistics on our long-term MODIS collection. We can start with the mean and standard deviation:

In [ ]:
modis_mean = modis.reduce(ee.Reducer.mean())

visualization_modis = {'min':0, 'max':0.75, 'palette':['DB670D','0DDB25'], 'alpha':0.75} 
Map = geemap.Map(center=[28.16514023225985, 83.66062796221756], zoom=6) 
Map.addLayer(modis_mean, visualization_modis) #Plot the map
Map

In [ ]:
modis_std = modis.reduce(ee.Reducer.stdDev())

visualization_modis = {'min':0, 'max':0.25, 'palette':['0D3ADB','C92442'], 'alpha':0.75} 
Map = geemap.Map(center=[28.16514023225985, 83.66062796221756], zoom=6) 
Map.addLayer(modis_std, visualization_modis) #Plot the map
Map

This is quite quick! This is 25 years of 16-day (so, ~26 times per year) data, at 250 m spatial resolution.

We can get a little fancier and compare individual years using *filters*:

In [ ]:
#Use filters to define two periods
modis_mean1 = modis.filterDate('2001-01-01','2003-01-01').reduce(ee.Reducer.mean()) #This is the earlier period (2001-2003)
modis_mean2 = modis.filterDate('2010-01-01','2012-01-01').reduce(ee.Reducer.mean()) #This is the later period (2010-2012)
visualization_modis = {'min':0, 'max':0.75, 'palette':['DB670D','0DDB25'], 'alpha':0.75} 

left_layer = geemap.ee_tile_layer(modis_mean1, visualization_modis, "Early")
right_layer = geemap.ee_tile_layer(modis_mean2, visualization_modis, "Late")

Map = geemap.Map(center=[28.16514023225985, 83.66062796221756], zoom=6) 
Map.split_map(left_layer, right_layer)
Map

There are differences, but they are quite small. We can emphasize the differences by directly comparing them:

In [ ]:
modis_dif = modis_mean2.subtract(modis_mean1) #Subtract the early data from the late data to get a difference map
visualization_modis = {'min':-0.25, 'max':0.25, 'palette':['C92442','FFFFFF','0D3ADB'], 'alpha':0.75} #Blue is 'greener', red is 'less green' 
Map = geemap.Map(center=[28.16514023225985, 83.66062796221756], zoom=6) 
Map.addLayer(modis_dif, visualization_modis) #Plot the map
Map

This is a quick and effective way to compare data! We can also do the same kinds of operations seasonally (summer vs winter) or over different time periods. 

Let's repeat this analysis with snow cover to see how that looks. I am selecting a single winter season of snow covered area ([https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD10A1](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD10A1)) in each case and doing a split-screen comparison:

In [ ]:
snow_cover = ee.ImageCollection("MODIS/061/MOD10A1").select('NDSI_Snow_Cover')

modis_mean1 = snow_cover.filterDate('2001-11-01','2002-03-01').reduce(ee.Reducer.mean()) #Early period
modis_mean2 = snow_cover.filterDate('2010-11-01','2011-03-01').reduce(ee.Reducer.mean()) #Late Period

modis_mean1 = modis_mean1.updateMask(modis_mean1.gt(0.1)) #Mask out non-snow covered areas
modis_mean2 = modis_mean2.updateMask(modis_mean2.gt(0.1))

visualization_modis = {'min':0, 'max':75, 'palette':['30D1CC','101AB0'], 'alpha':0.75} 

left_layer = geemap.ee_tile_layer(modis_mean1, visualization_modis, "2001")
right_layer = geemap.ee_tile_layer(modis_mean2, visualization_modis, "2010")

Map = geemap.Map(center=[28.16514023225985, 83.66062796221756], zoom=6) 
Map.split_map(left_layer, right_layer)
Map

In [ ]:
modis_dif = modis_mean2.subtract(modis_mean1)
visualization_modis = {'min':-10, 'max':10, 'palette':['C92442','FFFFFF','0D3ADB'], 'alpha':0.75} #Blue is 'more snow', red is 'less snow' 
Map = geemap.Map(center=[28.16514023225985, 83.66062796221756], zoom=6) 
Map.addLayer(modis_dif, visualization_modis) #Plot the map
Map

In this way we can quickly see which years had relatively less or more snow. You can repeat this kind of analysis for any data set in the Earth Engine catalog: [https://developers.google.com/earth-engine/datasets](https://developers.google.com/earth-engine/datasets), depending on what kind of data you are interested in.

## Visualizing Data Trends

To go beyond simple year-to-year comparisons, you can look at the rate of change in a data set. This is what is referred to as *trend analysis*. Fortunately, this is straightfoward in Earth Engine, and easy to accomplish using online processing.

Let's look at some simple climate data from ERA5: [https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_MONTHLY](https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_MONTHLY)

In [ ]:
monthly =  ee.ImageCollection("ECMWF/ERA5/MONTHLY") #Load in the monthly ERA5 data
monthly_length = monthly.size().getInfo()
print('Monthly', monthly_length)

In [ ]:
def to_mm(image): #Conversion tool
    mm_band = image.select('total_precipitation').multiply(1000).set('system:time_start', image.get('system:time_start'))
    return image.addBands(mm_band.rename('total_precipitation_mm'))
    
def time_regression(collection, variable, spacing='year'): #Regression tool for trend
    def addtime(image):
        date = image.date()
        years = date.difference(ee.Date('1900-01-01'), spacing)
        return image.addBands(ee.Image(years).rename('t')).float().addBands(ee.Image.constant(1))

    prepped = collection.map(addtime) #Add the new time band
    to_process = prepped.select(['t', variable]) #Choose what variable to regress against time
    
    return to_process.reduce(ee.Reducer.linearFit())

monthly_mm = monthly.map(to_mm) #Convert meters to mm to make the regression more stable
precip_annual_trend = time_regression(monthly_mm, 'total_precipitation_mm').select('scale') #Calculate the annual trend in mm precipitation

In [ ]:
#Plot total rainfall trends over the last decades (positive/negative)
image_viz_params = {'palette': ['B40426', 'E6745B', 'F6B69B', 'DDDDDD', 'A9C5FC', '6F91F2', '3B4CC0'],'min': -1,'max': 1, 'opacity':0.9}

#Blue is more rainfall, red is less rainfall
Map = geemap.Map(center=[28.16514023225985, 83.66062796221756], zoom=6) 
Map.addLayer(precip_annual_trend, image_viz_params)

Map

We can try the same analysis with the MODIS data as well:

In [ ]:
modis = ee.ImageCollection("MODIS/061/MOD13A1").select('NDVI')
modis = rescale(modis, 0.0001) #Rescale all of the images to the proper range (see here: https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13A1#bands)

modis = rescale(modis, 100) #Rescale all of the images a second time for the trend regression to work out (too small numbers make the trends hard to see)
ndvi_annual_trend = time_regression(modis, 'NDVI').select('scale') #Calculate the annual trend in NDVI

In [ ]:
image_viz_params = {'palette': ['B40426', 'E6745B', 'F6B69B', 'DDDDDD', 'A9C5FC', '6F91F2', '3B4CC0'],'min': -1,'max': 1, 'opacity':0.9}

#Blue is greener, red is less green
Map = geemap.Map(center=[28.16514023225985, 83.66062796221756], zoom=6) 
Map.addLayer(ndvi_annual_trend, image_viz_params)

Map

Now we can not only compare single years, but also look at linear trends in different variables! 